# CARD-style Correlation-Aware Restoration with Guided Diffusion

This Colab notebook moves the project closer to the CARD paper methodology. It uses an inference-only pretrained OpenAI guided-diffusion model as the image prior, then adds a covariance-aware measurement guidance term for correlated noise.

This is still not a line-for-line reproduction of the paper's DDRM closed-form updates. It is a practical Colab implementation of the same central idea:

1. model spatially correlated sensor noise with a covariance matrix,
2. whiten residuals using that covariance,
3. run diffusion inference with a data-consistency term that respects the covariance.

Run this notebook in Google Colab with **Runtime > Change runtime type > GPU**.


## 1. Install and download model assets inside Colab

The notebook uses OpenAI's `guided-diffusion` repository and the public `64x64_diffusion.pt` checkpoint. This smaller checkpoint is used so inference is practical in a course/demo Colab session.


In [ ]:
# Colab setup. Run this cell in Google Colab, not locally.
!git clone -q https://github.com/openai/guided-diffusion.git /content/guided-diffusion || true
!pip -q install blobfile tqdm matplotlib pillow

import sys
from pathlib import Path

sys.path.append('/content/guided-diffusion')
Path('/content/models').mkdir(parents=True, exist_ok=True)

!wget -nc https://openaipublic.blob.core.windows.net/diffusion/jul-2021/64x64_diffusion.pt -O /content/models/64x64_diffusion.pt


## 2. Imports and device


In [ ]:
import math
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm

from guided_diffusion.script_util import create_model_and_diffusion, model_and_diffusion_defaults

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


## 3. Load the pretrained guided-diffusion model

The flags below follow the OpenAI guided-diffusion README for the 64x64 model. The checkpoint is class-conditional, so the sampler receives an ImageNet class label. The label mostly acts as a weak natural-image prior for this restoration demo.


In [ ]:
def load_guided_diffusion_64(device: torch.device):
    options = model_and_diffusion_defaults()
    options.update({
        'attention_resolutions': '32,16,8',
        'class_cond': True,
        'diffusion_steps': 1000,
        'dropout': 0.1,
        'image_size': 64,
        'learn_sigma': True,
        'noise_schedule': 'cosine',
        'num_channels': 192,
        'num_head_channels': 64,
        'num_res_blocks': 3,
        'resblock_updown': True,
        'use_new_attention_order': True,
        'use_fp16': False,
        'use_scale_shift_norm': True,
        'timestep_respacing': '100',
    })
    model, diffusion = create_model_and_diffusion(**options)
    state = torch.load('/content/models/64x64_diffusion.pt', map_location='cpu')
    model.load_state_dict(state)
    model.to(device).eval()
    return model, diffusion

diffusion_model, diffusion = load_guided_diffusion_64(device)
print('model loaded')


## 4. Create or upload a clean image

The default path creates a synthetic RGB image. If you want to use your own image, set `USE_UPLOAD = True` and run the cell.


In [ ]:
USE_UPLOAD = False
IMAGE_SIZE = 64

def make_synthetic_rgb(size: int = 64) -> np.ndarray:
    coords = np.linspace(0.0, 1.0, size, dtype=np.float32)
    x_grid, y_grid = np.meshgrid(coords, coords)
    base = 0.35 * x_grid + 0.25 * y_grid
    disk = ((x_grid - 0.68) ** 2 + (y_grid - 0.35) ** 2) < 0.09
    square = (x_grid > 0.12) & (x_grid < 0.38) & (y_grid > 0.58) & (y_grid < 0.82)
    stripes = 0.12 * (np.sin(34 * x_grid) > 0)
    image = np.stack([base + stripes, base * 0.85 + stripes, base * 0.75], axis=-1)
    image[disk, :] += np.array([0.35, 0.33, 0.28], dtype=np.float32)
    image[square, :] -= np.array([0.25, 0.20, 0.12], dtype=np.float32)
    return np.clip(image, 0.0, 1.0).astype(np.float32)

def load_uploaded_image(size: int = 64) -> np.ndarray:
    from google.colab import files
    uploaded = files.upload()
    name = next(iter(uploaded))
    image = Image.open(name).convert('RGB').resize((size, size), Image.BICUBIC)
    return np.asarray(image, dtype=np.float32) / 255.0

clean_np = load_uploaded_image(IMAGE_SIZE) if USE_UPLOAD else make_synthetic_rgb(IMAGE_SIZE)
plt.figure(figsize=(3, 3))
plt.imshow(clean_np)
plt.axis('off')
plt.title('Clean image')
plt.show()


## 5. Correlated noise covariance, coloring, and whitening

This section mirrors CARD's noise model. We create a local patch covariance matrix, use it to color i.i.d. Gaussian noise, and keep the whitening matrix for covariance-aware data consistency during diffusion sampling.


In [ ]:
@dataclass(frozen=True)
class NoiseModel:
    patch_size: int
    covariance: torch.Tensor
    whitening: torch.Tensor
    coloring: torch.Tensor

def make_rolling_shutter_covariance(patch_size: int, rho: float, eps: float) -> np.ndarray:
    row, col = np.indices((patch_size, patch_size), dtype=np.float32).reshape(2, -1)
    row_distance = np.abs(row[:, None] - row[None, :])
    col_distance = np.abs(col[:, None] - col[None, :])
    covariance = (rho ** row_distance) * ((0.45 * rho) ** col_distance)
    covariance += eps * np.eye(patch_size * patch_size, dtype=np.float32)
    return covariance.astype(np.float32)

def build_noise_model(patch_size: int = 8, rho: float = 0.92, eps: float = 1e-5) -> NoiseModel:
    covariance_np = make_rolling_shutter_covariance(patch_size, rho, eps)
    eigvals, eigvecs = np.linalg.eigh(covariance_np)
    eigvals = np.maximum(eigvals, 1e-8)
    whitening_np = eigvecs @ np.diag(1.0 / np.sqrt(eigvals)) @ eigvecs.T
    coloring_np = eigvecs @ np.diag(np.sqrt(eigvals)) @ eigvecs.T
    return NoiseModel(
        patch_size=patch_size,
        covariance=torch.tensor(covariance_np, device=device),
        whitening=torch.tensor(whitening_np, dtype=torch.float32, device=device),
        coloring=torch.tensor(coloring_np, dtype=torch.float32, device=device),
    )

noise_model = build_noise_model()
identity_check = noise_model.whitening @ noise_model.covariance @ noise_model.whitening.T
print('whitened covariance mean absolute error:', (identity_check - torch.eye(identity_check.shape[0], device=device)).abs().mean().item())


In [ ]:
def image_to_tensor01(image: np.ndarray) -> torch.Tensor:
    return torch.tensor(image, dtype=torch.float32, device=device).permute(2, 0, 1).unsqueeze(0)

def tensor01_to_image(tensor: torch.Tensor) -> np.ndarray:
    array = tensor.detach().clamp(0, 1).squeeze(0).permute(1, 2, 0).cpu().numpy()
    return np.clip(array, 0.0, 1.0)

def extract_patch_matrix(image01: torch.Tensor, patch_size: int) -> torch.Tensor:
    patches = F.unfold(image01, kernel_size=patch_size, stride=patch_size)
    batch, channels_times_pixels, count = patches.shape
    channels = image01.shape[1]
    return patches.view(batch, channels, patch_size * patch_size, count)

def fold_patch_matrix(patches: torch.Tensor, height: int, width: int, patch_size: int) -> torch.Tensor:
    batch, channels, pixels, count = patches.shape
    merged = patches.view(batch, channels * pixels, count)
    return F.fold(merged, output_size=(height, width), kernel_size=patch_size, stride=patch_size)

def add_correlated_noise(clean01: torch.Tensor, model: NoiseModel, sigma: float, seed: int = 7) -> torch.Tensor:
    generator = torch.Generator(device=device).manual_seed(seed)
    patches = extract_patch_matrix(clean01, model.patch_size)
    white = torch.randn(patches.shape, generator=generator, device=device)
    colored = torch.einsum('ij,bcjn->bcin', model.coloring, white)
    noisy_patches = patches + sigma * colored
    return fold_patch_matrix(noisy_patches, clean01.shape[-2], clean01.shape[-1], model.patch_size).clamp(0, 1)

clean01 = image_to_tensor01(clean_np)
NOISE_SIGMA = 0.12
noisy01 = add_correlated_noise(clean01, noise_model, NOISE_SIGMA)

fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(tensor01_to_image(clean01)); axes[0].set_title('Clean'); axes[0].axis('off')
axes[1].imshow(tensor01_to_image(noisy01)); axes[1].set_title('Correlated noisy'); axes[1].axis('off')
plt.show()


## 6. CARD-style covariance-aware diffusion guidance

OpenAI guided-diffusion exposes a `cond_fn` hook for guidance. We use that hook to add a covariance-whitened measurement consistency gradient. For denoising, the forward operator is identity, so the residual is between the current diffusion sample and the noisy observation.

This is closer to the paper than the CPU-only project because the image prior is now a pretrained diffusion model. It is still an approximation because the original CARD paper modifies DDRM's closed-form updates directly.


In [ ]:
def to_model_range(image01: torch.Tensor) -> torch.Tensor:
    return image01 * 2.0 - 1.0

def from_model_range(image11: torch.Tensor) -> torch.Tensor:
    return ((image11 + 1.0) / 2.0).clamp(0, 1)

def whitened_residual_loss(sample11: torch.Tensor, observed01: torch.Tensor, model: NoiseModel) -> torch.Tensor:
    sample01 = from_model_range(sample11)
    residual = observed01 - sample01
    patches = extract_patch_matrix(residual, model.patch_size)
    whitened = torch.einsum('ij,bcjn->bcin', model.whitening, patches)
    return (whitened ** 2).mean()

GUIDANCE_SCALE = 65.0
CLASS_LABEL = 207  # ImageNet class conditioning; change this if desired.

def make_cond_fn(observed01: torch.Tensor, model: NoiseModel):
    def cond_fn(x: torch.Tensor, t: torch.Tensor, y: torch.Tensor | None = None) -> torch.Tensor:
        with torch.enable_grad():
            x_in = x.detach().requires_grad_(True)
            loss = whitened_residual_loss(x_in, observed01, model)
            grad = torch.autograd.grad(loss, x_in)[0]
        return -GUIDANCE_SCALE * grad
    return cond_fn


In [ ]:
def run_card_guided_diffusion(noisy01: torch.Tensor, noise_model: NoiseModel) -> torch.Tensor:
    y = torch.tensor([CLASS_LABEL], dtype=torch.long, device=device)
    model_kwargs = {'y': y}
    init = to_model_range(noisy01) + 0.25 * torch.randn_like(noisy01)
    init = init.clamp(-1, 1)
    sample = diffusion.p_sample_loop(
        diffusion_model,
        shape=init.shape,
        noise=init,
        clip_denoised=True,
        model_kwargs=model_kwargs,
        cond_fn=make_cond_fn(noisy01, noise_model),
        device=device,
        progress=True,
    )
    return from_model_range(sample)

restored01 = run_card_guided_diffusion(noisy01, noise_model)


## 7. Metrics and visualization


In [ ]:
def psnr(reference: np.ndarray, candidate: np.ndarray) -> float:
    mse = float(np.mean((reference - candidate) ** 2))
    return 99.0 if mse <= 0 else 10.0 * math.log10(1.0 / mse)

def global_ssim(reference: np.ndarray, candidate: np.ndarray) -> float:
    c1 = 0.01 ** 2
    c2 = 0.03 ** 2
    ref_mean = float(np.mean(reference))
    cand_mean = float(np.mean(candidate))
    ref_var = float(np.var(reference))
    cand_var = float(np.var(candidate))
    cov = float(np.mean((reference - ref_mean) * (candidate - cand_mean)))
    return ((2 * ref_mean * cand_mean + c1) * (2 * cov + c2)) / ((ref_mean ** 2 + cand_mean ** 2 + c1) * (ref_var + cand_var + c2))

noisy_np = tensor01_to_image(noisy01)
restored_np = tensor01_to_image(restored01)

print('Noisy    PSNR:', round(psnr(clean_np, noisy_np), 2), 'SSIM:', round(global_ssim(clean_np, noisy_np), 3))
print('Restored PSNR:', round(psnr(clean_np, restored_np), 2), 'SSIM:', round(global_ssim(clean_np, restored_np), 3))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for axis, title, image in zip(axes, ['Clean', 'Correlated noisy', 'CARD-guided diffusion'], [clean_np, noisy_np, restored_np]):
    axis.imshow(image)
    axis.set_title(title)
    axis.axis('off')
plt.tight_layout()
plt.show()


## 8. Save outputs


In [ ]:
Image.fromarray((restored_np * 255).astype(np.uint8)).save('/content/card_guided_diffusion_restored.png')
Image.fromarray((noisy_np * 255).astype(np.uint8)).save('/content/card_correlated_noisy.png')
Image.fromarray((clean_np * 255).astype(np.uint8)).save('/content/card_clean.png')
print('Saved:')
print('/content/card_clean.png')
print('/content/card_correlated_noisy.png')
print('/content/card_guided_diffusion_restored.png')


## Notes

- Increase `timestep_respacing` from `100` to `250` or `1000` for higher-quality but slower inference.
- Tune `GUIDANCE_SCALE` if the result is too noisy or too over-constrained.
- The 64x64 model is used for practicality. The OpenAI repository also provides 256x256 and 512x512 checkpoints, but those require more GPU memory and runtime.
- To become even closer to the CARD paper, the next step would be implementing DDRM's closed-form spectral updates in the whitened measurement space instead of using gradient guidance through `cond_fn`.
